# Module 14 - Preference tuning (DPO)

Use this notebook after `tests/test_dpo.py` is passing and after you have saved a Module 13 `*-SFT` model artifact. The notebook loads the strongest available SFT artifact by default, builds a small preference dataset, checks the step-0 `log(2)` invariant, trains with DPO, compares SFT vs DPO behavior, and saves a DPO artifact for Module 15.

DPO should feel like SFT with one extra dimension: every prompt has a chosen answer and a rejected answer, and the frozen reference model anchors how far the policy is allowed to move.

## Setup

In [1]:
from pathlib import Path
import json
import math
import subprocess
import sys

import matplotlib.pyplot as plt
import torch

from g2c.artifacts import (
    available_model_artifacts_with_suffix,
    best_model_artifact_with_suffix,
    load_model_artifact_with_tokenizer,
    save_huggingface_model_artifact,
    save_model_artifact,
)
from g2c.dpo import (
    DPOTrainer,
    PreferenceExample,
    dpo_loss,
    pad_and_collate_pref,
    sequence_logprob,
)
from g2c.notebook_extras.dpo import plot_dpo_history, train_dpo_with_progress
from g2c.sampling import generate
from g2c.sft import ChatTemplate

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

/Users/colkitt/sith/toys/courses/g2c


Run the DPO tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 14 TODOs in `g2c/dpo/`.

In [2]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_dpo.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 14 DPO tests are not passing yet."

..FFFFFFFFFFFFFFFFFFFFFFFFFFF........FFFFF.                              [100%]
=================================== FAILURES ===================================
________________ test_pad_and_collate_pref_returns_six_tensors _________________

    def test_pad_and_collate_pref_returns_six_tensors():
        examples = [_ex([1, 2], [3, 4], [5, 6, 7])]
>       out = pad_and_collate_pref(examples, max_seq_len=6, pad_id=0)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

tests/test_dpo.py:85: 
_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ 

examples = [PreferenceExample(prompt_ids=[1, 2], chosen_ids=[3, 4], rejected_ids=[5, 6, 7])]

    def pad_and_collate_pref(
        examples: list[PreferenceExample],
        *,
        max_seq_len: int,
        pad_id: int,
    ) -> tuple[
        torch.Tensor, torch.Tensor, torch.Tensor,
        torch.Tensor, torch.Tensor, torch.Tensor,
    ]:
        """Pad a list of preference examples and assem

AssertionError: Module 14 DPO tests are not passing yet.

## Load an SFT artifact

By default this loads the strongest saved `*-SFT` artifact from Module 13. Set `SFT_ARTIFACT_NAME` if you want to force a specific artifact such as `StoryLM-5M-SFT`, `TinyLLM-30M-SFT`, or `BaseLM-SFT`.

DPO holds two model copies at once: a trainable policy and a frozen reference. If you are using a large external BaseLM artifact and memory is tight, choose a smaller artifact or set `TRAIN_DEVICE = "cpu"` for the first debugging pass.

In [ ]:
SFT_ARTIFACT_NAME = None  # e.g. "StoryLM-5M-SFT", "TinyLLM-30M-SFT", "BaseLM-SFT"
TRAIN_DEVICE = "auto"
SEED = 14

available_sft = available_model_artifacts_with_suffix("-SFT", repo_root=repo_root)
if available_sft:
    print("available SFT artifacts:")
    for artifact in available_sft:
        print(f"  rank {artifact.rank:>3}: {artifact.name}")
else:
    print("No SFT artifacts found under artifacts/models/.")

if SFT_ARTIFACT_NAME is None:
    candidate = best_model_artifact_with_suffix("-SFT", repo_root=repo_root)
    if candidate is None:
        raise RuntimeError("Run Module 13 and save an SFT artifact before starting Module 14.")
    SFT_ARTIFACT_NAME = candidate.name

policy_artifact = load_model_artifact_with_tokenizer(
    SFT_ARTIFACT_NAME,
    repo_root=repo_root,
    device=TRAIN_DEVICE,
)
reference_artifact = load_model_artifact_with_tokenizer(
    SFT_ARTIFACT_NAME,
    repo_root=repo_root,
    device=TRAIN_DEVICE,
)

policy_model = policy_artifact.model
ref_model = reference_artifact.model
tokenizer = policy_artifact.tokenizer
template = ChatTemplate()
pad_id = tokenizer.special_to_id.get("<|pad|>", getattr(tokenizer, "pad_token_id", None) or 0)
end_id = tokenizer.special_to_id.get(template.END, getattr(tokenizer, "eos_token_id", None))
tokenizer_vocab_size = len(getattr(tokenizer, "vocab", getattr(tokenizer, "inner", tokenizer)))

def model_device(model) -> torch.device:
    device = getattr(model, "device", None)
    if isinstance(device, torch.device):
        return device
    for parameter in model.parameters():
        return parameter.device
    return torch.device("cpu")


print("loaded SFT artifact:", policy_artifact.name)
print("display:", policy_artifact.display_name)
print("kind:", policy_artifact.manifest.get("kind", "course_transformer"))
print("model vocab:", policy_model.vocab_size)
print("tokenizer vocab:", tokenizer_vocab_size)
print("max seq len:", policy_model.max_seq_len)
print("pad id:", pad_id, "end id:", end_id)
print("policy device:", model_device(policy_model))

## Sampling helpers

These are notebook helpers, not the Module 14 deliverable. They render the same chat template used in SFT and stop generation at `<|end|>` when that token exists.

In [ ]:
def chat_prompt(user_text: str, assistant_prefix: str = "") -> str:
    return (
        template.render([{"role": "user", "content": user_text}])
        + f"{template.ASSISTANT}\n"
        + assistant_prefix
    )


def encode_for_model(model, text: str) -> torch.Tensor:
    ids = tokenizer.encode_with_vocab_size(text, model.vocab_size)
    if not ids:
        raise ValueError("prompt encoded to no tokens")
    return torch.tensor(ids, dtype=torch.long)


def sample_from_model(
    model,
    prompt: str,
    *,
    max_new_tokens: int = 80,
    temperature: float = 0.7,
    top_p: float | None = 0.9,
    seed: int = SEED,
) -> str:
    prompt_ids = encode_for_model(model, prompt)
    ids = generate(
        model,
        prompt_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.1,
        eos_id=end_id,
        generator=torch.Generator().manual_seed(seed),
    )
    return tokenizer.decode([int(x) for x in ids.tolist()])


def sample_response(model, user_text: str, **kwargs) -> str:
    return sample_from_model(model, chat_prompt(user_text), **kwargs)


def printable(text: str) -> str:
    has_control = any(ord(ch) < 32 and ch not in "\n\t" for ch in text)
    return text.encode("unicode_escape").decode("ascii") if has_control else text


def show_response(label: str, model, user_text: str, **kwargs) -> str:
    text = sample_response(model, user_text, **kwargs)
    print("-" * 72)
    print(label)
    print(printable(text))
    return text

A quick baseline read on the SFT model. This is not graded; it gives you a qualitative anchor before DPO changes the policy.

In [ ]:
BASELINE_PROMPTS = [
    "What is the capital of Spain?",
    "Answer in one short sentence: what is a neural network?",
    "If you are unsure, how should you answer?",
]

for prompt in BASELINE_PROMPTS:
    print("=" * 72)
    print("user:", prompt)
    show_response("SFT model", ref_model, prompt, max_new_tokens=80, seed=SEED)

## Exercise 1 - Build preference pairs

Start with the compact starter set below, then add your own rows. A serious deliverable should have closer to 100 preference pairs, but this starter set is enough to exercise the DPO loop and inspect whether the model moves in the intended direction.

Each row should vary one axis at a time. Avoid pairing a long polished chosen answer with a short broken rejected answer unless the preference you want to teach is length.

In [ ]:
preference_rows = [
    {"kind": "factual", "user": "What is the capital of France?", "chosen": "Paris.", "rejected": "London."},
    {"kind": "factual", "user": "What is the capital of Spain?", "chosen": "Madrid.", "rejected": "Lisbon."},
    {"kind": "factual", "user": "What is the capital of Italy?", "chosen": "Rome.", "rejected": "Milan."},
    {"kind": "factual", "user": "What is the capital of Germany?", "chosen": "Berlin.", "rejected": "Munich."},
    {"kind": "factual", "user": "What is the capital of Japan?", "chosen": "Tokyo.", "rejected": "Kyoto."},
    {"kind": "factual", "user": "Which planet is known as the red planet?", "chosen": "Mars.", "rejected": "Venus."},
    {"kind": "factual", "user": "What gas do plants take in for photosynthesis?", "chosen": "Carbon dioxide.", "rejected": "Oxygen."},
    {"kind": "factual", "user": "What do bees make?", "chosen": "Honey.", "rejected": "Milk."},
    {"kind": "arithmetic", "user": "What is 2 + 3?", "chosen": "5.", "rejected": "6."},
    {"kind": "arithmetic", "user": "What is 4 * 5?", "chosen": "20.", "rejected": "9."},
    {"kind": "arithmetic", "user": "What is 12 - 7?", "chosen": "5.", "rejected": "19."},
    {"kind": "arithmetic", "user": "What is 8 + 6?", "chosen": "14.", "rejected": "13."},
    {"kind": "opposites", "user": "Answer in one word: opposite of hot.", "chosen": "Cold.", "rejected": "Warm."},
    {"kind": "opposites", "user": "Answer in one word: opposite of up.", "chosen": "Down.", "rejected": "Side."},
    {"kind": "opposites", "user": "Answer in one word: opposite of early.", "chosen": "Late.", "rejected": "Soon."},
    {"kind": "opposites", "user": "Answer in one word: opposite of empty.", "chosen": "Full.", "rejected": "Open."},
    {"kind": "style", "user": "Explain a tensor in one sentence.", "chosen": "A tensor is an array of numbers with a shape.", "rejected": "A tensor is a thing that is very important and can be many things in many ways."},
    {"kind": "style", "user": "Explain gradient descent in one sentence.", "chosen": "Gradient descent updates parameters in the direction that lowers loss.", "rejected": "Gradient descent is when the model goes around and learns stuff until it gets better somehow."},
    {"kind": "style", "user": "Explain a tokenizer in one sentence.", "chosen": "A tokenizer converts text into token IDs a model can process.", "rejected": "A tokenizer is a big complicated text thing that does all sorts of text processing."},
    {"kind": "style", "user": "Explain logits in one sentence.", "chosen": "Logits are unnormalized scores before softmax turns them into probabilities.", "rejected": "Logits are numbers and then there is softmax and then it is kind of probability-like."},
    {"kind": "helpfulness", "user": "Give one tip for debugging a failing test.", "chosen": "Run the smallest failing test and inspect the first wrong value.", "rejected": "Tests are annoying, so just change the code until it works."},
    {"kind": "helpfulness", "user": "What should I do if my training loss is NaN?", "chosen": "Lower the learning rate and check for unstable operations or invalid data.", "rejected": "Keep training; NaN usually fixes itself."},
    {"kind": "helpfulness", "user": "How do I check whether a model is overfitting?", "chosen": "Compare train loss against validation loss over time.", "rejected": "Only look at the final training loss."},
    {"kind": "helpfulness", "user": "What should I do before changing many hyperparameters?", "chosen": "Change one thing at a time and record the result.", "rejected": "Change everything at once so the run is different."},
    {"kind": "honesty", "user": "If you are not sure about an answer, what should you say?", "chosen": "I am not sure.", "rejected": "I know for certain."},
    {"kind": "honesty", "user": "If a fact might be outdated, what should an assistant do?", "chosen": "Say it may need verification.", "rejected": "State it confidently anyway."},
    {"kind": "honesty", "user": "Should an assistant invent a source when it lacks one?", "chosen": "No, it should say it does not have a source.", "rejected": "Yes, it should make one sound plausible."},
    {"kind": "honesty", "user": "If the prompt is ambiguous, what is a good response?", "chosen": "Ask a clarifying question or state the assumption.", "rejected": "Guess silently and pretend it was clear."},
    {"kind": "format", "user": "Return exactly one color.", "chosen": "Blue.", "rejected": "Blue is a color that many people like."},
    {"kind": "format", "user": "Return exactly one number: seven.", "chosen": "7.", "rejected": "The number is 7, which is seven."},
    {"kind": "format", "user": "Answer yes or no: is water wet?", "chosen": "Yes.", "rejected": "Water can be considered wet in many contexts."},
    {"kind": "format", "user": "Answer with one word: cat or dog?", "chosen": "Cat.", "rejected": "I would choose cat because cats are nice."},
]

print(f"preference rows: {len(preference_rows)}")
print(json.dumps(preference_rows[0], indent=2))

Save your preference rows if you want the dataset as a reusable artifact. The notebook keeps this disabled by default so exploratory edits do not overwrite a hand-curated file.

In [ ]:
SAVE_PREFERENCE_JSON = False
PREFERENCE_JSON_PATH = repo_root / "data" / "dpo" / "preferences.json"

if SAVE_PREFERENCE_JSON:
    PREFERENCE_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    PREFERENCE_JSON_PATH.write_text(
        json.dumps(preference_rows, indent=2) + "\n",
        encoding="utf-8",
    )
    print("saved", PREFERENCE_JSON_PATH.relative_to(repo_root))
else:
    print("not saved; set SAVE_PREFERENCE_JSON = True when the dataset is ready")

Check token lengths before training. If chosen completions are systematically longer, DPO can learn length instead of preference.

In [ ]:
def response_ids(text: str) -> list[int]:
    return tokenizer.encode_with_vocab_size(text + template.END, policy_model.vocab_size)

length_rows = []
for row in preference_rows:
    chosen_len = len(response_ids(row["chosen"]))
    rejected_len = len(response_ids(row["rejected"]))
    ratio = max(chosen_len, rejected_len) / max(1, min(chosen_len, rejected_len))
    length_rows.append((row["kind"], chosen_len, rejected_len, ratio, row["user"]))

print(f"{'kind':<12} {'chosen':>6} {'rejected':>8} {'ratio':>6}  prompt")
for kind, chosen_len, rejected_len, ratio, user in length_rows[:20]:
    print(f"{kind:<12} {chosen_len:>6} {rejected_len:>8} {ratio:>6.2f}  {user[:54]}")

avg_chosen = sum(row[1] for row in length_rows) / len(length_rows)
avg_rejected = sum(row[2] for row in length_rows) / len(length_rows)
print("\naverage chosen tokens:", round(avg_chosen, 2))
print("average rejected tokens:", round(avg_rejected, 2))
print("max length ratio:", round(max(row[3] for row in length_rows), 2))

## Encode preference triples

A DPO prompt is the user turn plus the assistant role marker. The chosen and rejected responses are completions after that marker, and both include `<|end|>` so the model can learn which answer should stop.

In [ ]:
def render_dpo_prompt(user_text: str) -> str:
    return template.render([{"role": "user", "content": user_text}]) + f"{template.ASSISTANT}\n"


def encode_preference(row: dict) -> PreferenceExample:
    prompt_ids = tokenizer.encode_with_vocab_size(
        render_dpo_prompt(row["user"]),
        policy_model.vocab_size,
    )
    chosen_ids = tokenizer.encode_with_vocab_size(
        row["chosen"] + template.END,
        policy_model.vocab_size,
    )
    rejected_ids = tokenizer.encode_with_vocab_size(
        row["rejected"] + template.END,
        policy_model.vocab_size,
    )
    return PreferenceExample(
        prompt_ids=prompt_ids,
        chosen_ids=chosen_ids,
        rejected_ids=rejected_ids,
    )

encoded_rows = [(row, encode_preference(row)) for row in preference_rows]
encoded_preferences = [ex for _, ex in encoded_rows]

first_row, first_ex = encoded_rows[0]
print("prompt text:")
print(render_dpo_prompt(first_row["user"]))
print("prompt ids:", len(first_ex.prompt_ids))
print("chosen ids:", len(first_ex.chosen_ids), tokenizer.decode(first_ex.chosen_ids))
print("rejected ids:", len(first_ex.rejected_ids), tokenizer.decode(first_ex.rejected_ids))
assert max(max(ex.prompt_ids + ex.chosen_ids + ex.rejected_ids) for ex in encoded_preferences) < policy_model.vocab_size

Inspect the collator on two examples. The mask should be `1` only on response targets, never on the prompt or padding.

In [ ]:
DPO_MAX_SEQ_LEN = min(128, policy_model.max_seq_len)

cx, cy, cm, rx, ry, rm = pad_and_collate_pref(
    encoded_preferences[:2],
    max_seq_len=DPO_MAX_SEQ_LEN,
    pad_id=pad_id,
)
print("chosen x/y/mask:", cx.shape, cy.shape, cm.shape)
print("rejected x/y/mask:", rx.shape, ry.shape, rm.shape)
print("chosen mask sums:", cm.sum(dim=1).tolist())
print("rejected mask sums:", rm.sum(dim=1).tolist())


def masked_target_text(y: torch.Tensor, mask: torch.Tensor) -> str:
    ids = [int(token_id) for token_id, keep in zip(y.tolist(), mask.tolist()) if int(keep) == 1]
    return tokenizer.decode(ids)

print("\nchosen target text:", masked_target_text(cy[0], cm[0]))
print("rejected target text:", masked_target_text(ry[0], rm[0]))

## Train/validation split

In [ ]:
generator = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(encoded_rows), generator=generator).tolist()
split = max(1, int(0.8 * len(perm)))
train_indices = perm[:split]
val_indices = perm[split:] or perm[-1:]

train_rows = [encoded_rows[i][0] for i in train_indices]
val_rows = [encoded_rows[i][0] for i in val_indices]
train_examples = [encoded_rows[i][1] for i in train_indices]
val_examples = [encoded_rows[i][1] for i in val_indices]

print("train examples:", len(train_examples))
print("val examples:", len(val_examples))

## Exercise 2 - Step-0 DPO sanity check

Before training, the policy and reference are identical copies. The DPO margin is therefore zero and the loss should be `log(2) ≈ 0.6931`. This is the deepest single sanity check for your data path.

In [ ]:
def preference_batch_metrics(policy, reference, examples, *, beta: float, max_seq_len: int):
    cx, cy, cm, rx, ry, rm = pad_and_collate_pref(
        examples,
        max_seq_len=max_seq_len,
        pad_id=pad_id,
    )
    device = model_device(policy)
    cx, cy, cm = cx.to(device), cy.to(device), cm.to(device)
    rx, ry, rm = rx.to(device), ry.to(device), rm.to(device)
    with torch.no_grad():
        policy_c = sequence_logprob(policy(cx), cy, cm)
        policy_r = sequence_logprob(policy(rx), ry, rm)
        ref_c = sequence_logprob(reference(cx), cy, cm)
        ref_r = sequence_logprob(reference(rx), ry, rm)
        loss, metrics = dpo_loss(policy_c, policy_r, ref_c, ref_r, beta=beta)
    return {
        "loss": loss.item(),
        "chosen_reward": metrics["chosen_reward"].item(),
        "rejected_reward": metrics["rejected_reward"].item(),
        "reward_margin": metrics["reward_margin"].item(),
        "accuracy": metrics["accuracy"].item(),
    }

initial_metrics = preference_batch_metrics(
    policy_model,
    ref_model,
    train_examples[: min(8, len(train_examples))],
    beta=0.1,
    max_seq_len=DPO_MAX_SEQ_LEN,
)
print(initial_metrics)
print("log(2):", math.log(2))
assert abs(initial_metrics["loss"] - math.log(2)) < 1e-3

## Exercise 3 - Train DPO

The defaults are intentionally conservative. DPO is about 3x the wall-clock of SFT at the same model size because it runs policy/reference on chosen/rejected sequences. Increase steps only after the loss, reward margin, and samples make sense.

In [ ]:
IS_HF_ARTIFACT = policy_artifact.manifest.get("kind") == "huggingface_causal_lm"

DPO_CONFIG = {
    "max_seq_len": DPO_MAX_SEQ_LEN,
    "pad_id": pad_id,
    "beta": 0.1,
    "batch_size": 1 if IS_HF_ARTIFACT else 4,
    "max_steps": 120 if IS_HF_ARTIFACT else 300,
    "max_lr": 5e-5 if IS_HF_ARTIFACT else 1e-4,
    "min_lr": 5e-6 if IS_HF_ARTIFACT else 1e-5,
    "warmup_steps": 10,
    "weight_decay": 0.0,
    "grad_clip": 1.0,
    "eval_every": 20 if IS_HF_ARTIFACT else 50,
    "eval_iters": 3,
    "log_every": 5 if IS_HF_ARTIFACT else 10,
    "device": TRAIN_DEVICE,
}
DPO_CONFIG

In [ ]:
trainer = DPOTrainer(
    policy_model,
    ref_model=ref_model,
    examples=train_examples,
    generator=torch.Generator().manual_seed(SEED),
    **DPO_CONFIG,
)

history = train_dpo_with_progress(
    f"{policy_artifact.name} DPO",
    trainer,
    eval_examples=val_examples,
)
plot_dpo_history(history)

## Exercise 4 - Compare SFT vs DPO behavior

Use the same prompts, seed, and sampling settings. The reference model is your original SFT checkpoint; the policy model is now DPO-tuned.

In [ ]:
COMPARISON_PROMPTS = [
    "What is the capital of Spain?",
    "What is the capital of France?",
    "Explain logits in one sentence.",
    "If you are unsure about an answer, what should you say?",
    "Return exactly one color.",
]


def compare_models(prompts: list[str], *, seed: int = SEED) -> None:
    for prompt in prompts:
        print("=" * 72)
        print("user:", prompt)
        show_response("SFT/reference", ref_model, prompt, max_new_tokens=80, seed=seed)
        show_response("DPO/policy", policy_model, prompt, max_new_tokens=80, seed=seed)

compare_models(COMPARISON_PROMPTS)

Score held-out preference pairs directly. Positive reward margin means the DPO policy favors the chosen completion more than the frozen SFT reference does.

In [ ]:
def score_preference_example(policy, reference, ex: PreferenceExample, *, beta: float = 0.1) -> dict[str, float]:
    metrics = preference_batch_metrics(policy, reference, [ex], beta=beta, max_seq_len=DPO_MAX_SEQ_LEN)
    return metrics

print(f"{'kind':<12} {'margin':>8} {'acc':>5}  prompt")
for row, ex in zip(val_rows, val_examples):
    metrics = score_preference_example(policy_model, ref_model, ex, beta=DPO_CONFIG["beta"])
    print(f"{row['kind']:<12} {metrics['reward_margin']:>8.3f} {metrics['accuracy']:>5.2f}  {row['user'][:52]}")

### Written reflection

Question: Where did DPO clearly improve the SFT model, and where did it fail or make behavior worse?

Answer: 

## Exercise 5 - Optional beta sweep

Enable this only after the baseline run works. The quickest useful sweep is three betas over fewer steps. The goal is not a perfect model; it is to see low beta drift, mid beta learning, and high beta under-movement.

In [ ]:
RUN_BETA_SWEEP = False
BETA_VALUES = [0.05, 0.1, 0.3]
BETA_SWEEP_STEPS = 80

beta_results = {}
if RUN_BETA_SWEEP:
    for beta in BETA_VALUES:
        fresh_policy = load_model_artifact_with_tokenizer(
            SFT_ARTIFACT_NAME,
            repo_root=repo_root,
            device=TRAIN_DEVICE,
        ).model
        fresh_ref = load_model_artifact_with_tokenizer(
            SFT_ARTIFACT_NAME,
            repo_root=repo_root,
            device=TRAIN_DEVICE,
        ).model
        config = {**DPO_CONFIG, "beta": beta, "max_steps": BETA_SWEEP_STEPS}
        sweep_trainer = DPOTrainer(
            fresh_policy,
            ref_model=fresh_ref,
            examples=train_examples,
            generator=torch.Generator().manual_seed(SEED),
            **config,
        )
        sweep_history = train_dpo_with_progress(
            f"beta={beta}",
            sweep_trainer,
            eval_examples=val_examples,
        )
        beta_results[beta] = {
            "history": sweep_history,
            "model": fresh_policy,
        }

    for beta, result in beta_results.items():
        h = result["history"]
        print(
            beta,
            "final loss", round(h["train_loss"][-1], 4),
            "final margin", round(h["reward_margin"][-1], 4),
            "final acc", round(h["accuracy"][-1], 3),
        )

### Beta sweep notes

Question: Which beta moved the model enough without visibly damaging its base behavior?

Answer: 

## Exercise 6 - Save the DPO artifact

This preserves the original SFT artifact and writes a separate DPO artifact for Module 15 evaluation. If your samples got worse, set `SAVE_DPO_ARTIFACT = False`, tune the run, and save only when the artifact is worth reusing.

In [ ]:
DPO_ARTIFACT_NAME = (
    SFT_ARTIFACT_NAME[:-4] + "-DPO"
    if SFT_ARTIFACT_NAME.endswith("-SFT")
    else f"{SFT_ARTIFACT_NAME}-DPO"
)
SAVE_DPO_ARTIFACT = True

if SAVE_DPO_ARTIFACT:
    training_config = {
        **DPO_CONFIG,
        "sft_artifact": SFT_ARTIFACT_NAME,
        "num_examples": len(encoded_preferences),
        "num_train_examples": len(train_examples),
        "num_val_examples": len(val_examples),
        "preference_rows": len(preference_rows),
    }
    if policy_artifact.manifest.get("kind") == "huggingface_causal_lm":
        artifact_dir = save_huggingface_model_artifact(
            DPO_ARTIFACT_NAME,
            model=policy_model,
            tokenizer=tokenizer,
            base_artifact_name=SFT_ARTIFACT_NAME,
            training_config=training_config,
            source=f"DPO on {len(encoded_preferences)} preference pairs from {SFT_ARTIFACT_NAME}",
            history=history,
            module="module-14",
            notes="Preference-tuned checkpoint for Module 15 evaluation experiments.",
            repo_root=repo_root,
        )
    else:
        model_config = dict(policy_artifact.manifest["model_config"])
        artifact_dir = save_model_artifact(
            DPO_ARTIFACT_NAME,
            model=policy_model,
            model_config=model_config,
            training_config=training_config,
            tokenizer_artifact_name=policy_artifact.manifest["tokenizer_artifact"],
            source=f"DPO on {len(encoded_preferences)} preference pairs from {SFT_ARTIFACT_NAME}",
            history=history,
            seed=SEED,
            module="module-14",
            notes="Preference-tuned checkpoint for Module 15 evaluation experiments.",
            repo_root=repo_root,
        )
    print("saved", artifact_dir.relative_to(repo_root))
else:
    print("not saved")

## Deliverable notes

Question: Explain why the initial DPO loss is `log(2)`.

Answer: 

Question: Explain why the reference model must stay frozen.

Answer: 

Question: What is one preference-dataset bias you checked for?

Answer: 